# 01 — Preprocessing: CHIRPS Rainfall + NOAA ONI

**Author:** Tim Mangsakala  
**Purpose:** Load raw data dari CHIRPS (GEE export) dan NOAA ONI, clean, validate, dan export ke `data/processed/` siap untuk feature engineering.

## Inputs
- `data/raw/kebumen_chirps_daily_1995_2025.csv` — output dari GEE CHIRPS export
- `data/raw/oni_monthly_raw.csv` — manual download dari NOAA CPC

## Outputs
- `data/processed/rainfall_daily_clean.csv` — daily rainfall Kebumen 1995-2025
- `data/processed/oni_monthly.csv` — monthly ONI dengan lag features

## Validation Checks
1. Date range coverage (no gaps)
2. Missing values < 5%
3. Outlier detection (rainfall > 200mm/day flag)
4. ENSO phase classification matches NOAA standard

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Paths (assume notebook runs from notebooks/)
DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Raw data dir: {DATA_RAW.absolute()}")
print(f"Processed data dir: {DATA_PROCESSED.absolute()}")
print(f"Files in raw: {list(DATA_RAW.glob('*.csv'))}")

## Part 1: CHIRPS Rainfall Loading

GEE export kasih CSV dengan kolom: `system:index, date, rainfall_mm, .geo`. Kita cuma butuh `date` dan `rainfall_mm`.

In [ ]:
# Load CHIRPS export
rainfall_raw = pd.read_csv(DATA_RAW / 'kebumen_chirps_daily_1995_2025.csv')

print(f"Shape: {rainfall_raw.shape}")
print(f"Columns: {rainfall_raw.columns.tolist()}")
rainfall_raw.head()

In [ ]:
# Clean: keep only date and rainfall, parse date
rainfall = rainfall_raw[['date', 'rainfall_mm']].copy()
rainfall['date'] = pd.to_datetime(rainfall['date'])
rainfall = rainfall.sort_values('date').reset_index(drop=True)

# Check date range
print(f"Date range: {rainfall['date'].min()} → {rainfall['date'].max()}")
print(f"Total days: {len(rainfall)}")
print(f"Expected days (1995-01-01 to 2025-12-31): {(pd.Timestamp('2025-12-31') - pd.Timestamp('1995-01-01')).days + 1}")

In [ ]:
# Validation 1: Check for missing dates
expected_dates = pd.date_range(start='1995-01-01', end='2025-12-31', freq='D')
actual_dates = set(rainfall['date'])
missing_dates = set(expected_dates) - actual_dates

print(f"Missing dates: {len(missing_dates)}")
if missing_dates:
    print(f"Sample missing: {sorted(list(missing_dates))[:10]}")
else:
    print("✅ No missing dates")

In [ ]:
# Validation 2: Missing values
n_null = rainfall['rainfall_mm'].isna().sum()
pct_null = (n_null / len(rainfall)) * 100
print(f"Null rainfall values: {n_null} ({pct_null:.2f}%)")

if n_null > 0:
    # Forward fill small gaps (max 3 consecutive days)
    rainfall['rainfall_mm'] = rainfall['rainfall_mm'].fillna(method='ffill', limit=3)
    # Remaining nulls: fill with 0 (assume dry)
    rainfall['rainfall_mm'] = rainfall['rainfall_mm'].fillna(0)
    print(f"After fill: {rainfall['rainfall_mm'].isna().sum()} nulls remaining")

In [ ]:
# Validation 3: Outlier detection
extreme_days = rainfall[rainfall['rainfall_mm'] > 200]
print(f"Days with rainfall > 200mm (extreme): {len(extreme_days)}")
if len(extreme_days) > 0:
    print("Sample extreme events:")
    print(extreme_days.nlargest(5, 'rainfall_mm'))

# Negative rainfall (data error)
negative = rainfall[rainfall['rainfall_mm'] < 0]
if len(negative) > 0:
    print(f"⚠️ Negative rainfall found: {len(negative)} rows. Setting to 0.")
    rainfall.loc[rainfall['rainfall_mm'] < 0, 'rainfall_mm'] = 0

In [ ]:
# Quick visualization: annual total rainfall
rainfall['year'] = rainfall['date'].dt.year
annual_total = rainfall.groupby('year')['rainfall_mm'].sum()

fig, ax = plt.subplots(figsize=(12, 4))
annual_total.plot(kind='bar', ax=ax, color='#5B8AA6')
ax.axhline(annual_total.mean(), color='#C8553D', linestyle='--', label=f'Mean: {annual_total.mean():.0f}mm')
ax.set_title('Annual Total Rainfall — Kebumen (1995-2025)')
ax.set_ylabel('Rainfall (mm)')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../docs/annual_rainfall.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\nAnnual rainfall stats:")
print(f"  Mean: {annual_total.mean():.0f}mm")
print(f"  Min: {annual_total.min():.0f}mm ({annual_total.idxmin()})")
print(f"  Max: {annual_total.max():.0f}mm ({annual_total.idxmax()})")

In [ ]:
# Save cleaned rainfall data
rainfall_clean = rainfall[['date', 'rainfall_mm']].copy()
rainfall_clean.to_csv(DATA_PROCESSED / 'rainfall_daily_clean.csv', index=False)
print(f"✅ Saved: {DATA_PROCESSED / 'rainfall_daily_clean.csv'}")
print(f"   Shape: {rainfall_clean.shape}")

## Part 2: NOAA ONI Loading

**Manual prep:** Download dari https://origin.cpc.ncep.noaa.gov/products/analysis_monitoring/ensostuff/ONI_v5.php

Tabel ONI di NOAA punya format wide: kolom = season (DJF, JFM, FMA, ...), row = year. Kita perlu convert ke long format.

**Format input expected** (`oni_monthly_raw.csv`):
```
Year,DJF,JFM,FMA,MAM,AMJ,MJJ,JJA,JAS,ASO,SON,OND,NDJ
1995,1.1,0.7,0.4,0.2,0.0,-0.1,-0.2,-0.5,-0.8,-0.9,-0.9,-0.8
...
```

Kalau format lo beda, sesuaikan parsing di bawah.

In [ ]:
# Load ONI raw
oni_raw = pd.read_csv(DATA_RAW / 'oni_monthly_raw.csv')
print(f"Shape: {oni_raw.shape}")
oni_raw.head()

In [ ]:
# Convert wide to long
# ONI seasons: DJF=center on Jan, JFM=Feb, FMA=Mar, MAM=Apr, AMJ=May, MJJ=Jun,
# JJA=Jul, JAS=Aug, ASO=Sep, SON=Oct, OND=Nov, NDJ=Dec
season_to_month = {
    'DJF': 1, 'JFM': 2, 'FMA': 3, 'MAM': 4, 'AMJ': 5, 'MJJ': 6,
    'JJA': 7, 'JAS': 8, 'ASO': 9, 'SON': 10, 'OND': 11, 'NDJ': 12
}

# Melt dataframe
oni_long = oni_raw.melt(
    id_vars=['Year'],
    var_name='season',
    value_name='oni_value'
)
oni_long['month'] = oni_long['season'].map(season_to_month)
oni_long = oni_long.rename(columns={'Year': 'year'})
oni_long = oni_long[['year', 'month', 'oni_value', 'season']].sort_values(['year', 'month']).reset_index(drop=True)

print(f"Long format shape: {oni_long.shape}")
print(f"Year range: {oni_long['year'].min()} - {oni_long['year'].max()}")
oni_long.head(13)

In [ ]:
# Filter to range 1994-2025 (1994 needed for lag-3 features in 1995)
oni = oni_long[(oni_long['year'] >= 1994) & (oni_long['year'] <= 2025)].copy()

# Validation: no nulls
print(f"Null ONI values: {oni['oni_value'].isna().sum()}")

# Compute lag features
oni['oni_lag_3'] = oni['oni_value'].shift(3)
oni['oni_lag_6'] = oni['oni_value'].shift(6)

# Drop first 6 months (no lag-6 available) — start clean from Jul 1994
oni_with_lag = oni.dropna(subset=['oni_lag_6']).reset_index(drop=True)
print(f"After lag computation: {oni_with_lag.shape}")

In [ ]:
# Classify ENSO phase per NOAA standard
def classify_enso(oni_val):
    if pd.isna(oni_val):
        return 'unknown'
    if oni_val >= 2.0:
        return 'el_nino_strong'
    elif oni_val >= 1.5:
        return 'el_nino_moderate'
    elif oni_val >= 0.5:
        return 'el_nino_weak'
    elif oni_val > -0.5:
        return 'neutral'
    elif oni_val > -1.5:
        return 'la_nina_weak'
    elif oni_val > -2.0:
        return 'la_nina_moderate'
    else:
        return 'la_nina_strong'

oni_with_lag['enso_phase'] = oni_with_lag['oni_value'].apply(classify_enso)

# Summary
phase_counts = oni_with_lag['enso_phase'].value_counts()
print("ENSO Phase Distribution:")
print(phase_counts)

In [ ]:
# Visualize ONI time series with phase coloring
fig, ax = plt.subplots(figsize=(14, 4))

# Plot line
ax.plot(pd.to_datetime(oni_with_lag[['year', 'month']].assign(day=1)),
         oni_with_lag['oni_value'], color='gray', linewidth=0.5)

# Fill above/below thresholds
dates = pd.to_datetime(oni_with_lag[['year', 'month']].assign(day=1))
ax.fill_between(dates, oni_with_lag['oni_value'], 0.5,
                where=(oni_with_lag['oni_value'] > 0.5), color='#C8553D', alpha=0.5, label='El Niño')
ax.fill_between(dates, oni_with_lag['oni_value'], -0.5,
                where=(oni_with_lag['oni_value'] < -0.5), color='#5B8AA6', alpha=0.5, label='La Niña')

ax.axhline(0.5, color='red', linestyle='--', alpha=0.3)
ax.axhline(-0.5, color='blue', linestyle='--', alpha=0.3)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('ONI Time Series 1994-2025 — ENSO Phases')
ax.set_ylabel('ONI Value')
ax.legend()
plt.tight_layout()
plt.savefig('../docs/oni_timeseries.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Save processed ONI
oni_final = oni_with_lag[['year', 'month', 'oni_value', 'oni_lag_3', 'oni_lag_6', 'enso_phase']].copy()
oni_final.to_csv(DATA_PROCESSED / 'oni_monthly.csv', index=False)
print(f"✅ Saved: {DATA_PROCESSED / 'oni_monthly.csv'}")
print(f"   Shape: {oni_final.shape}")
oni_final.head(10)

## Part 3: Sanity Check — Combined View

Quick check rainfall pattern di tahun El Niño vs La Niña vs Neutral untuk verifikasi data make sense.

In [ ]:
# Get JJAS average ONI per year (proxy untuk dry season ENSO state)
jjas_oni = oni_final[oni_final['month'].isin([6, 7, 8, 9])].groupby('year')['oni_value'].mean()

# Get OND total rainfall per year (wet season onset period)
rainfall_clean['year'] = rainfall_clean['date'].dt.year
rainfall_clean['month'] = rainfall_clean['date'].dt.month
ond_rain = rainfall_clean[rainfall_clean['month'].isin([10, 11, 12])].groupby('year')['rainfall_mm'].sum()

# Combine
combined = pd.DataFrame({'jjas_oni': jjas_oni, 'ond_rainfall': ond_rain}).dropna()

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(combined['jjas_oni'], combined['ond_rainfall'],
                      c=combined['jjas_oni'], cmap='RdBu_r', s=80,
                      edgecolor='black', linewidth=0.5)

# Annotate extreme years
for year in [1997, 2015, 1998, 2010]:
    if year in combined.index:
        ax.annotate(str(year),
                    (combined.loc[year, 'jjas_oni'], combined.loc[year, 'ond_rainfall']),
                    fontsize=9, xytext=(5, 5), textcoords='offset points')

ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('JJAS ONI (ENSO state during dry season)')
ax.set_ylabel('OND Total Rainfall (mm) — wet season onset period')
ax.set_title('Sanity Check: ENSO vs Wet Season Rainfall (Kebumen)\nExpect: positive ONI → less rainfall (delayed onset)')
plt.colorbar(scatter, label='JJAS ONI')
plt.tight_layout()
plt.savefig('../docs/enso_rainfall_correlation.png', dpi=100, bbox_inches='tight')
plt.show()

# Compute correlation
corr = combined['jjas_oni'].corr(combined['ond_rainfall'])
print(f"\nCorrelation JJAS ONI vs OND rainfall: {corr:.3f}")
print("Expected: negative correlation (El Niño → drier wet season)")

## Summary

✅ Outputs ready:
- `data/processed/rainfall_daily_clean.csv` — daily rainfall, no nulls, validated
- `data/processed/oni_monthly.csv` — monthly ONI dengan lag-3, lag-6, dan ENSO phase

**Next step:** Run `02_onset_detection.ipynb` untuk detect onset musim hujan tiap tahun.